# 🧠 Huấn Luyện AI Cờ Tướng (Xiangqi HalfKP NNUE)
### Kết nối Thư Viện Engine Nổi Tiếng (Pikafish, Fairy-Stockfish) & Huấn luyện Kỳ Phổ Đại Sư

Notebook này hướng dẫn và tự động hóa toàn bộ quy trình huấn luyện cho dự án Game Cờ Tướng:
1. **Tải & Cài đặt Engine ngoài**: Tải và thiết lập các Engine cờ tướng hàng đầu thế giới (**Pikafish** & **Fairy-Stockfish**) vào thư mục biệt lập `engines/` (được `.gitignore` bảo vệ).
2. **Huấn luyện Giám sát (Supervised Learning)**: Huấn luyện mạng NNUE HalfKP từ tập kỳ phổ mẫu của các đại kiện tướng.
3. **Chưng cất Tri thức (Knowledge Distillation)**: Dùng Pikafish / Fairy-Stockfish làm "Sư phụ" đánh giá và gán nhãn thế cờ.
4. **Đấu tập Đối kháng (Sparring)**: Cho Bot thi đấu trực tiếp với Engine ngoài để cọ xát thực chiến.
5. **Quản lý Checkpoints & Đóng gói WASM**: Lưu các mốc trọng số theo từng thế hệ vào `checkpoints/` (được `.gitignore` bảo vệ) và tự động biên dịch lại gói WebAssembly (`cotuong_core.wasm`) cho giao diện web.

--- 
## 🛠️ Bước 1: Kiểm Tra Môi Trường & Khởi Tạo Thư Mục Làm Việc

In [23]:
# Tạo các thư mục lưu trữ riêng (đã được cấu hình trong .gitignore để không đẩy lên GitHub)
!mkdir -p engines checkpoints

import os
import sys
import subprocess

print("🔍 Kiểm tra môi trường hệ thống:")
!rustc --version
!cargo --version
!node --version
!npm --version

cpu_cores = os.cpu_count()
print(f"✅ Số nhân CPU khả dụng: {cpu_cores}")
print(f"📁 Thư mục Engines    : {os.path.abspath('engines')}")
print(f"📁 Thư mục Checkpoints: {os.path.abspath('checkpoints')}")

🔍 Kiểm tra môi trường hệ thống:
rustc 1.96.0 (ac68faa20 2026-05-25)
cargo 1.96.0 (30a34c682 2026-05-25)
v22.22.2
10.9.7
✅ Số nhân CPU khả dụng: 16
📁 Thư mục Engines    : /home/exblackhole/Desktop/GameCoTuong/engines
📁 Thư mục Checkpoints: /home/exblackhole/Desktop/GameCoTuong/checkpoints


--- 
## 🐟 Bước 2: Tải & Cài Đặt Engine Pikafish (UCCI / UCI Protocol)
**Pikafish** là engine cờ tướng mạnh nhất thế giới hiện nay (dựa trên nền tảng Stockfish). Ở bước này, chúng ta sẽ thiết lập binary Pikafish vào thư mục `engines/pikafish/`.

In [24]:
%%bash
set -e
ENGINES_DIR="$(pwd)/engines"
PIKAFISH_DIR="$ENGINES_DIR/pikafish"
mkdir -p "$PIKAFISH_DIR"

TARGET_BIN="$PIKAFISH_DIR/pikafish"

if [ -f "$TARGET_BIN" ]; then
    echo "✅ Pikafish đã có sẵn tại: $TARGET_BIN"
else
    echo "⏳ Đang chuẩn bị Engine Pikafish..."
    cd "$PIKAFISH_DIR"
    
    # Thử tải bản prebuilt chính thức cho Linux x86-64
    DOWNLOAD_SUCCESS=0
    RELEASE_URL=$(curl -s https://api.github.com/repos/official-pikafish/Pikafish/releases/latest | grep "browser_download_url.*pikafish-.*x86-64.*\.zip" | head -n 1 | cut -d '"' -f 4 || true)
    
    if [ -n "$RELEASE_URL" ]; then
        echo "⬇️ Đang tải bản phát hành từ: $RELEASE_URL"
        curl -L -o pikafish_release.zip "$RELEASE_URL"
        unzip -o pikafish_release.zip
        rm -f pikafish_release.zip
        # Tìm file binary vừa giải nén
        FOUND_BIN=$(find . -maxdepth 2 -type f -name "pikafish*" ! -name "*.zip" ! -name "*.nnue" | head -n 1)
        if [ -n "$FOUND_BIN" ]; then
            chmod +x "$FOUND_BIN"
            cp "$FOUND_BIN" "$TARGET_BIN"
            DOWNLOAD_SUCCESS=1
        fi
    fi
    
    # Nếu không tải được release hoặc máy offline, biên dịch nhanh từ GitHub repo
    if [ "$DOWNLOAD_SUCCESS" -eq 0 ]; then
        echo "🛠️ Đang biên dịch Pikafish từ nguồn..."
        rm -rf /tmp/pikafish_src
        git clone --depth 1 https://github.com/official-pikafish/Pikafish.git /tmp/pikafish_src
        make -C /tmp/pikafish_src/src -j$(nproc) build ARCH=x86-64-modern
        cp /tmp/pikafish_src/src/pikafish "$TARGET_BIN"
        chmod +x "$TARGET_BIN"
        rm -rf /tmp/pikafish_src
    fi
fi

# Kiểm thử phản hồi giao thức UCCI của Pikafish
echo "🧪 Kiểm thử giao thức UCCI với Pikafish..."
echo -e "ucci\nquit" | "$TARGET_BIN" | head -n 6
echo "✅ Pikafish đã sẵn sàng tại: $TARGET_BIN"

✅ Pikafish đã có sẵn tại: /home/exblackhole/Desktop/GameCoTuong/engines/pikafish/pikafish
🧪 Kiểm thử giao thức UCCI với Pikafish...
Pikafish dev-20261029-fec5d430 by the Pikafish developers (see AUTHORS file)
Unknown command: 'ucci'. Type help for more information.
✅ Pikafish đã sẵn sàng tại: /home/exblackhole/Desktop/GameCoTuong/engines/pikafish/pikafish


--- 
## 🧚 Bước 3: Tải & Cài Đặt Engine Fairy-Stockfish (UCI Protocol)
**Fairy-Stockfish** là engine nguồn mở chuyên về các biến thể cờ tướng phương Đông và cờ vua phương Tây, hỗ trợ giao thức UCI chuẩn mực với tùy chọn `setoption name UCI_Variant value xiangqi`.

In [25]:
%%bash
set -e
ENGINES_DIR="$(pwd)/engines"
FAIRY_DIR="$ENGINES_DIR/fairy-stockfish"
mkdir -p "$FAIRY_DIR"

TARGET_BIN="$FAIRY_DIR/fairy-stockfish"

if [ -f "$TARGET_BIN" ]; then
    echo "✅ Fairy-Stockfish đã có sẵn tại: $TARGET_BIN"
else
    echo "⏳ Đang chuẩn bị Engine Fairy-Stockfish..."
    cd "$FAIRY_DIR"
    
    DOWNLOAD_SUCCESS=0
    # Thử tải bản phát hành prebuilt cho largeboard x86-64
    RELEASE_URL=$(curl -s https://api.github.com/repos/fairy-stockfish/Fairy-Stockfish/releases/latest | grep "browser_download_url.*largeboard_x86-64-modern" | head -n 1 | cut -d '"' -f 4 || true)
    
    if [ -n "$RELEASE_URL" ]; then
        echo "⬇️ Đang tải từ: $RELEASE_URL"
        curl -L -o "$TARGET_BIN" "$RELEASE_URL"
        chmod +x "$TARGET_BIN"
        DOWNLOAD_SUCCESS=1
    fi
    
    # Nếu không tải được prebuilt, biên dịch nhanh từ nguồn
    if [ "$DOWNLOAD_SUCCESS" -eq 0 ]; then
        echo "🛠️ Đang biên dịch Fairy-Stockfish từ nguồn..."
        rm -rf /tmp/fairy_src
        git clone --depth 1 https://github.com/fairy-stockfish/Fairy-Stockfish.git /tmp/fairy_src
        make -C /tmp/fairy_src/src -j$(nproc) build ARCH=x86-64-modern
        cp /tmp/fairy_src/src/fairy-stockfish "$TARGET_BIN"
        chmod +x "$TARGET_BIN"
        rm -rf /tmp/fairy_src
    fi
fi

# Kiểm thử phản hồi giao thức UCI với biến thể cờ tướng
echo "🧪 Kiểm thử giao thức UCI với Fairy-Stockfish..."
echo -e "uci\nsetoption name UCI_Variant value xiangqi\nisready\nquit" | "$TARGET_BIN" | grep -E "id name|readyok"
echo "✅ Fairy-Stockfish đã sẵn sàng tại: $TARGET_BIN"

✅ Fairy-Stockfish đã có sẵn tại: /home/exblackhole/Desktop/GameCoTuong/engines/fairy-stockfish/fairy-stockfish
🧪 Kiểm thử giao thức UCI với Fairy-Stockfish...
id name Fairy-Stockfish 14 LB
readyok
✅ Fairy-Stockfish đã sẵn sàng tại: /home/exblackhole/Desktop/GameCoTuong/engines/fairy-stockfish/fairy-stockfish


--- 
## 📖 Bước 4: Huấn Luyện Giám Sát từ Kỳ Phổ Mẫu (Supervised Learning)
Đây là bước quan trọng nhất: Huấn luyện mạng NNUE học ngay từ các thế cờ danh thủ trong `cotuong_trainer/data/sample_master_games.txt` thay vì tự đấu mò mẫm từ đầu.

- **Đầu vào**: Tệp kỳ phổ mẫu.
- **Kết quả**: Cập nhật trọng số active sang `cotuong_core/assets/default.nnue`.
- **Checkpoints**: Lưu từng thế hệ vào thư mục `checkpoints/`.

In [26]:
# Chạy huấn luyện Supervised Learning 5 Epochs (hoặc thay đổi số epochs tùy ý)
!chmod +x train.sh
!./train.sh --mode dataset --epochs 5 --eval-games 10 --checkpoints checkpoints

 🧠 Bắt đầu Huấn Luyện AI Cờ Tướng (NNUE HalfKP Master Dataset & External Engine)
🎯 Cấu hình:
  • Kiến trúc mạng           : HalfKP(10.530 -> 192) x 2 -> 32 -> 32 -> 1
  • File trọng số đích       : cotuong_core/assets/default.nnue
  • Tham số thực thi         : --mode dataset --epochs 5 --eval-games 10 --checkpoints checkpoints

▶ 1. Chạy quá trình huấn luyện AI (cotuong_trainer)...
   Compiling cotuong_core v0.1.0 (/home/exblackhole/Desktop/GameCoTuong/cotuong_core)
   Compiling cotuong_trainer v0.1.0 (/home/exblackhole/Desktop/GameCoTuong/cotuong_trainer)
    Finished `release` profile [optimized] target(s) in 7.96sr(bin)        
     Running `target/release/cotuong_trainer --output cotuong_core/assets/default.nnue --mode dataset --epochs 5 --eval-games 10 --checkpoints checkpoints`
 🧠 Xiangqi AI Trainer (Master Dataset & External UCCI/UCI Engine Integration)  
Cấu hình:
  • Chế độ (Mode)            : dataset
  • Số Epochs                : 5
  • Tốc độ học (LR)          : 0.01
  • Đư

--- 
## 👨‍🏫 Bước 5: Chưng Cất Tri Thức cùng Sư Phụ Pikafish (Knowledge Distillation)
Dùng Pikafish làm "Sư phụ" để gán nhãn điểm số (`search_eval` ở độ sâu 10–14). Mạng NNUE của dự án sẽ học theo phong cách đánh giá của siêu engine thế giới.

In [27]:
# Chạy chưng cất tri thức với Engine sư phụ Pikafish
!cargo run --release -p cotuong_trainer -- \
    --mode teacher \
    --engine engines/pikafish/pikafish \
    --proto ucci \
    --depth 10 \
    --epochs 5 \
    --checkpoints checkpoints

   Compiling cotuong_core v0.1.0 (/home/exblackhole/Desktop/GameCoTuong/cotuong_core)
   Compiling cotuong_trainer v0.1.0 (/home/exblackhole/Desktop/GameCoTuong/cotuong_trainer)
    Finished `release` profile [optimized] target(s) in 7.90sr(bin)        
     Running `target/release/cotuong_trainer --mode teacher --engine engines/pikafish/pikafish --proto ucci --depth 10 --epochs 5 --checkpoints checkpoints`
 🧠 Xiangqi AI Trainer (Master Dataset & External UCCI/UCI Engine Integration)  
Cấu hình:
  • Chế độ (Mode)            : teacher
  • Số Epochs                : 5
  • Tốc độ học (LR)          : 0.01
  • Đường dẫn trọng số đích : cotuong_core/assets/default.nnue
  • Thư mục Checkpoint       : checkpoints
  • Engine Sư Phụ            : engines/pikafish/pikafish (Ucci)
  • Độ sâu phân tích         : 10
  • Tập thế cờ gán nhãn      : cotuong_trainer/data/sample_master_games.txt

▶ Khởi động Engine Sư Phụ ("engines/pikafish/pikafish", protocol: Ucci) ...
✅ Đã kết nối thành công với Engine

--- 
## 🥊 Bước 6: Đấu Tập Đối Kháng Trực Tiếp với Engine Ngoài (Sparring Match)
Cho mô hình AI của dự án thi đấu đối kháng trực tiếp luân phiên cầm quân Đỏ và Đen với Pikafish hoặc Fairy-Stockfish để kiểm tra trình độ thực chiến.

In [28]:
# Đấu tập 10 ván với Engine ngoài (Pikafish / Fairy-Stockfish) ở độ sâu 8
!cargo run --release -p cotuong_trainer -- \
    --mode sparring \
    --engine engines/pikafish/pikafish \
    --proto ucci \
    --games 10 \
    --depth 8

   Compiling cotuong_core v0.1.0 (/home/exblackhole/Desktop/GameCoTuong/cotuong_core)
   Compiling cotuong_trainer v0.1.0 (/home/exblackhole/Desktop/GameCoTuong/cotuong_trainer)
    Finished `release` profile [optimized] target(s) in 7.89sr(bin)        
     Running `target/release/cotuong_trainer --mode sparring --engine engines/pikafish/pikafish --proto ucci --games 10 --depth 8`
 🧠 Xiangqi AI Trainer (Master Dataset & External UCCI/UCI Engine Integration)  
Cấu hình:
  • Chế độ (Mode)            : sparring
  • Số Epochs                : 5
  • Tốc độ học (LR)          : 0.01
  • Đường dẫn trọng số đích : cotuong_core/assets/default.nnue
  • Thư mục Checkpoint       : checkpoints
  • Engine Đối thủ           : engines/pikafish/pikafish (Ucci)
  • Số ván đấu               : 10
  • Độ sâu đấu tập           : 8

▶ Khởi tạo trận đấu tập (Sparring) với Engine: engines/pikafish/pikafish ...
✅ Đã sẵn sàng thi đấu 10 ván đối kháng!
  ▶ Ván 1/10: Bot cầm ĐỎ
  ▶ Ván 2/10: Bot cầm ĐEN
  ▶ Ván 3/

--- 
## 🗄️ Bước 7: Quản Lý Checkpoints & Đóng Gói WebAssembly (WASM)
Liệt kê tất cả các checkpoint đã lưu trong thư mục `checkpoints/`, xác minh tính toàn vẹn của mô hình và biên dịch gói WASM để giao diện Web sẵn sàng phục vụ người chơi.

In [29]:
import glob
import os

print("📦 Danh sách các Checkpoints đã lưu trong checkpoints/:")
checkpoints = sorted(glob.glob("checkpoints/*.nnue"))
if checkpoints:
    for cp in checkpoints:
        size_kb = os.path.getsize(cp) / 1024.0
        print(f"  • {cp} ({size_kb:.1f} KB)")
else:
    print("  (Chưa có file checkpoint nào, hãy chạy bước huấn luyện ở trên)")

print("\n▶ Trọng số chính đang dùng:")
default_nnue = "cotuong_core/assets/default.nnue"
if os.path.exists(default_nnue):
    print(f"  • {default_nnue} ({os.path.getsize(default_nnue) / 1024.0:.1f} KB)")

📦 Danh sách các Checkpoints đã lưu trong checkpoints/:
  • checkpoints/dataset_epoch_1.nnue (4127.0 KB)
  • checkpoints/dataset_epoch_3.nnue (4127.0 KB)
  • checkpoints/teacher_epoch_1.nnue (4127.0 KB)
  • checkpoints/teacher_epoch_2.nnue (4127.0 KB)
  • checkpoints/teacher_epoch_3.nnue (4127.0 KB)
  • checkpoints/teacher_epoch_4.nnue (4127.0 KB)
  • checkpoints/teacher_epoch_5.nnue (4127.0 KB)

▶ Trọng số chính đang dùng:
  • cotuong_core/assets/default.nnue (4127.0 KB)


In [30]:
# Kiểm thử tính hợp lệ và Accumulator của model NNUE
!cargo test -p cotuong_core -- test_embedded_weights_validity test_accumulator_incremental_vs_refresh

# Biên dịch lại gói WebAssembly (WASM)
!npm run build:wasm
print("\n🎉 Hoàn tất đóng gói! Bạn có thể chạy 'npm run dev' trên terminal để trải nghiệm AI trên trình duyệt.")

   Compiling cotuong_core v0.1.0 (/home/exblackhole/Desktop/GameCoTuong/cotuong_core)
    Finished `test` profile [unoptimized + debuginfo] target(s) in 0.36s   
     Running unittests src/lib.rs (target/debug/deps/cotuong_core-132c201bcd89cba7)

running 2 tests
test engine::nnue::tests::test_embedded_weights_validity ... ok
test engine::nnue::tests::test_accumulator_incremental_vs_refresh ... ok

test result: ok. 2 passed; 0 failed; 0 ignored; 0 measured; 33 filtered out; finished in 0.10s

     Running tests/mate_solving.rs (target/debug/deps/mate_solving-d1aa54a08addb97b)

running 0 tests

test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 4 filtered out; finished in 0.00s

   Doc-tests cotuong_core

running 0 tests

test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.00s


> game-co-tuong@0.1.0 build:wasm
> cargo build -p cotuong_core --target wasm32-unknown-unknown --release && wasm-bindgen --target web --out-dir src/wasm target/wasm3